# Silver do grupo - tabelas de correspondência (categorias e países)

Este notebook corre **depois** das Silver de todos os elementos e cria as duas tabelas que ligam os datasets entre si:

| Tabela Silver | 1 linha = | Usada no modelo em |
|---|---|---|
| `silver.mapa_categorias` | 1 categoria de origem de uma fonte (`fonte`, `categoria_origem`, `subcategoria_origem`) | `dim_categoria_produto` |
| `silver.paises` | 1 nome de país tal como aparece numa fonte | `dim_localizacao` |

**Categorias.** Cada dataset tem a sua classificação (Superstore: Furniture/Office Supplies/Technology; DataCo: departamentos
desportivos; Olist e funil: segmentos em inglês com `_`; Datafiniti e Amazon: etiquetas de eletrónica; Women's Clothing e
Rent the Runway: departamentos de roupa). Todas são traduzidas para uma lista comum de **áreas** e **categorias**
(ex.: Eletrónica > Áudio, Vestuário & Calçado > Roupa de Mulher). A subcategoria fica a da origem.
A Datafiniti tem mais de 200 etiquetas diferentes (algumas com texto lixo), por isso usa regras por palavras-chave.

**Países.** O DataCo tem os países em espanhol ("Francia", "Alemania") e o Superstore tem 7 nomes diferentes dos do
World Data ("Myanmar (Burma)", "Côte-d'Ivoire"...). Todos passam ao nome do World Data e ao código ISO.
Também se corrigem 7 códigos ISO que vinham vazios no World Data (ex.: a Namíbia tem o código "NA", lido como valor em falta).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - mapas do grupo (categorias e países)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

## Lista comum de áreas e categorias

In [3]:
ELE, VES, CASA, ESC, DESP, SAU, LIV, BEBE, ALI, ANI, AUTO, OUT = (
    "Eletrónica", "Vestuário & Calçado", "Casa & Mobiliário", "Escritório & Papelaria", "Desporto & Lazer",
    "Saúde & Beleza", "Livros, Música & Filmes", "Bebé", "Alimentação & Bebidas", "Animais", "Automóvel", "Outros")

# Eletrónica: regras por palavras-chave (usadas na Datafiniti e na Amazon). A ordem conta: a primeira regra que bate ganha.
REGRAS_ELETRONICA = [
    ("Videojogos", ["video game", "console", "game capture", "virtual reality", "gaming"]),
    ("Eletrónica Auto & GPS", ["car ", "car&", "vehicle", "gps", "auto &", "marine", "satellite radio", "sirius", "android auto", "dashcam"]),
    ("Telemóveis & Wearables", ["phone", "mobile", "iphone", "galaxy", "smart watch", "smartwatch", "wearable", "telephon"]),
    ("TV & Vídeo", ["tv", "television", "projector", "streaming", "blu-ray", "dvd", "video camcorder", "flat screen", "dtv"]),
    ("Fotografia & Vídeo", ["camera", "photo", "lens", "camcorder", "tripod", "flash", "drone", "instant film", "surveillance", "security"]),
    ("Áudio", ["headphone", "headset", "speaker", "audio", "stereo", "subwoofer", "sound", "amplifier", "receiver", "reciever",
               "ipod", "mp3", "turntable", "radio", "hi-fi", "microphone", "bose", "sennheiser", "jensen", "pioneer", "ihome", "echo"]),
    ("Computadores & Acessórios", ["computer", "laptop", "desktop", "memory", "drive", "ssd", "graphics", "tablet", "ipad",
                                   "networking", "wi-fi", "usb", "office", "keyboard", "ebook", "e-book", "tower", "monitor", "micro sd"]),
]

def categoria_eletronica(texto):
    t = (texto or "").lower()
    for cat, palavras in REGRAS_ELETRONICA:
        if any(p in t for p in palavras):
            return cat
    return "Outros Eletrónicos"

# Superstore: (categoria, subcategoria) -> (área, categoria comum)
SUPERSTORE = {
    ("Furniture", "Bookcases"): (CASA, "Mobiliário"), ("Furniture", "Chairs"): (CASA, "Mobiliário"),
    ("Furniture", "Tables"): (CASA, "Mobiliário"), ("Furniture", "Furnishings"): (CASA, "Decoração & Têxteis"),
    ("Office Supplies", "Appliances"): (ELE, "Eletrodomésticos"), ("Office Supplies", "Art"): (ESC, "Papelaria & Arte"),
    ("Technology", "Accessories"): (ELE, "Computadores & Acessórios"), ("Technology", "Phones"): (ELE, "Telemóveis & Wearables"),
    ("Technology", "Copiers"): (ESC, "Equipamento de Escritório"), ("Technology", "Machines"): (ESC, "Equipamento de Escritório"),
}
def mapa_superstore(cat, sub):
    return SUPERSTORE.get((cat, sub), (ESC, "Material de Escritório") if cat == "Office Supplies" else (OUT, "Outros"))

# DataCo: categoria (category_name) -> (área, categoria comum)
DATACO = {
    "Baby": (BEBE, "Bebé"), "Children's Clothing": (VES, "Roupa de Criança"), "Cleats": (VES, "Calçado"),
    "Crafts": (ESC, "Papelaria & Arte"), "Men's Clothing": (VES, "Roupa de Homem"), "Men's Footwear": (VES, "Calçado"),
    "Women's Clothing": (VES, "Roupa de Mulher"), "Books": (LIV, "Livros"), "CDs": (LIV, "Música & Filmes"),
    "DVDs": (LIV, "Música & Filmes"), "Music": (LIV, "Música & Filmes"), "Video Games": (ELE, "Videojogos"),
    "Camping & Hiking": (DESP, "Ar Livre"), "Fishing": (DESP, "Ar Livre"), "Hunting & Shooting": (DESP, "Ar Livre"),
    "Water Sports": (DESP, "Ar Livre"), "Indoor/Outdoor Games": (DESP, "Brinquedos & Jogos"), "Toys": (DESP, "Brinquedos & Jogos"),
    "Baseball & Softball": (DESP, "Desporto & Fitness"), "Basketball": (DESP, "Desporto & Fitness"), "Hockey": (DESP, "Desporto & Fitness"),
    "Lacrosse": (DESP, "Desporto & Fitness"), "Soccer": (DESP, "Desporto & Fitness"), "Sporting Goods": (DESP, "Desporto & Fitness"),
    "Tennis & Racquet": (DESP, "Desporto & Fitness"), "Boxing & MMA": (DESP, "Desporto & Fitness"),
    "Cardio Equipment": (DESP, "Desporto & Fitness"), "Fitness Accessories": (DESP, "Desporto & Fitness"),
    "Strength Training": (DESP, "Desporto & Fitness"), "Electronics": (ELE, "Outros Eletrónicos"),
    "Shop By Sport": (DESP, "Golfe"), "Women's Apparel": (DESP, "Golfe"), "Golf Apparel": (DESP, "Golfe"),
    "Golf Bags & Carts": (DESP, "Golfe"), "Golf Balls": (DESP, "Golfe"), "Golf Gloves": (DESP, "Golfe"), "Golf Shoes": (DESP, "Golfe"),
    "Kids' Golf Clubs": (DESP, "Golfe"), "Men's Golf Clubs": (DESP, "Golfe"), "Women's Golf Clubs": (DESP, "Golfe"),
    "Health and Beauty": (SAU, "Saúde & Beleza"), "Garden": (CASA, "Jardim & Bricolage"), "Pet Supplies": (ANI, "Animais"),
    "Cameras": (ELE, "Fotografia & Vídeo"), "Computers": (ELE, "Computadores & Acessórios"),
    "Consumer Electronics": (ELE, "Outros Eletrónicos"), "As Seen on TV!": (OUT, "Outros"), "Trade-In": (OUT, "Outros"),
}

# Olist (categoria em inglês) e funil (segmento de negócio): o mesmo vocabulário em snake_case
OLIST = {
    "agro_industry_and_commerce": (OUT, "Outros"), "air_conditioning": (ELE, "Eletrodomésticos"), "art": (ESC, "Papelaria & Arte"),
    "arts_and_craftmanship": (ESC, "Papelaria & Arte"), "audio": (ELE, "Áudio"), "auto": (AUTO, "Acessórios Auto"),
    "baby": (BEBE, "Bebé"), "bed_bath_table": (CASA, "Decoração & Têxteis"), "books_general_interest": (LIV, "Livros"),
    "books_imported": (LIV, "Livros"), "books_technical": (LIV, "Livros"), "cds_dvds_musicals": (LIV, "Música & Filmes"),
    "christmas_supplies": (CASA, "Decoração & Têxteis"), "cine_photo": (ELE, "Fotografia & Vídeo"),
    "computers": (ELE, "Computadores & Acessórios"), "computers_accessories": (ELE, "Computadores & Acessórios"),
    "consoles_games": (ELE, "Videojogos"), "construction_tools_construction": (CASA, "Jardim & Bricolage"),
    "construction_tools_lights": (CASA, "Jardim & Bricolage"), "construction_tools_safety": (CASA, "Jardim & Bricolage"),
    "cool_stuff": (OUT, "Outros"), "costruction_tools_garden": (CASA, "Jardim & Bricolage"),
    "costruction_tools_tools": (CASA, "Jardim & Bricolage"), "diapers_and_hygiene": (BEBE, "Bebé"),
    "drinks": (ALI, "Alimentação & Bebidas"), "dvds_blu_ray": (LIV, "Música & Filmes"), "electronics": (ELE, "Outros Eletrónicos"),
    "fashio_female_clothing": (VES, "Roupa de Mulher"), "fashion_bags_accessories": (VES, "Acessórios de Moda"),
    "fashion_childrens_clothes": (VES, "Roupa de Criança"), "fashion_male_clothing": (VES, "Roupa de Homem"),
    "fashion_shoes": (VES, "Calçado"), "fashion_sport": (VES, "Roupa Desportiva"), "fashion_underwear_beach": (VES, "Roupa de Mulher"),
    "fixed_telephony": (ELE, "Telemóveis & Wearables"), "flowers": (CASA, "Decoração & Têxteis"), "food": (ALI, "Alimentação & Bebidas"),
    "food_drink": (ALI, "Alimentação & Bebidas"), "furniture_bedroom": (CASA, "Mobiliário"), "furniture_decor": (CASA, "Decoração & Têxteis"),
    "furniture_living_room": (CASA, "Mobiliário"), "furniture_mattress_and_upholstery": (CASA, "Mobiliário"),
    "garden_tools": (CASA, "Jardim & Bricolage"), "health_beauty": (SAU, "Saúde & Beleza"), "home_appliances": (ELE, "Eletrodomésticos"),
    "home_appliances_2": (ELE, "Eletrodomésticos"), "home_comfort_2": (CASA, "Decoração & Têxteis"), "home_confort": (CASA, "Decoração & Têxteis"),
    "home_construction": (CASA, "Jardim & Bricolage"), "housewares": (CASA, "Cozinha & Utilidades"),
    "industry_commerce_and_business": (OUT, "Outros"), "kitchen_dining_laundry_garden_furniture": (CASA, "Mobiliário"),
    "la_cuisine": (CASA, "Cozinha & Utilidades"), "luggage_accessories": (VES, "Acessórios de Moda"), "market_place": (OUT, "Outros"),
    "music": (LIV, "Música & Filmes"), "musical_instruments": (DESP, "Instrumentos Musicais"), "office_furniture": (CASA, "Mobiliário"),
    "party_supplies": (CASA, "Decoração & Têxteis"), "perfumery": (SAU, "Perfumaria"), "pet_shop": (ANI, "Animais"),
    "security_and_services": (OUT, "Outros"), "signaling_and_security": (CASA, "Jardim & Bricolage"), "small_appliances": (ELE, "Eletrodomésticos"),
    "small_appliances_home_oven_and_coffee": (ELE, "Eletrodomésticos"), "sports_leisure": (DESP, "Desporto & Fitness"),
    "stationery": (ESC, "Material de Escritório"), "tablets_printing_image": (ELE, "Computadores & Acessórios"),
    "telephony": (ELE, "Telemóveis & Wearables"), "toys": (DESP, "Brinquedos & Jogos"), "watches_gifts": (VES, "Acessórios de Moda"),
    "furniture_mattress_and_upholstery": (CASA, "Mobiliário"), "fashion_female_clothing": (VES, "Roupa de Mulher"),
    "portable_kitchen_and_food_preparers": (ELE, "Eletrodomésticos"), "pc_gamer": (ELE, "Videojogos"),
    # segmentos do funil que não são categorias da Olist
    "audio_video_electronics": (ELE, "Áudio"), "bags_backpacks": (VES, "Acessórios de Moda"), "books": (LIV, "Livros"),
    "car_accessories": (AUTO, "Acessórios Auto"), "construction_tools_house_garden": (CASA, "Jardim & Bricolage"),
    "fashion_accessories": (VES, "Acessórios de Moda"), "food_supplement": (SAU, "Saúde & Beleza"), "games_consoles": (ELE, "Videojogos"),
    "gifts": (OUT, "Outros"), "handcrafted": (ESC, "Papelaria & Arte"), "home_decor": (CASA, "Decoração & Têxteis"),
    "home_office_furniture": (CASA, "Mobiliário"), "household_utilities": (CASA, "Cozinha & Utilidades"), "jewerly": (VES, "Acessórios de Moda"),
    "music_instruments": (DESP, "Instrumentos Musicais"), "other": (OUT, "Outros"), "party": (CASA, "Decoração & Têxteis"),
    "perfume": (SAU, "Perfumaria"), "pet": (ANI, "Animais"), "phone_mobile": (ELE, "Telemóveis & Wearables"),
    "religious": (OUT, "Outros"), "watches": (VES, "Acessórios de Moda"),
}

## Construção da `mapa_categorias`

Recolhem-se as categorias distintas de cada tabela Silver e aplica-se a correspondência acima.

In [4]:
def distintos(tabela, *cols):
    return [tuple(r) for r in spark.table(tabela).select(*cols).distinct().collect()]

linhas = []   # (fonte, categoria_origem, subcategoria_origem, area, categoria)
for cat, sub in distintos("silver.superstore_vendas", "categoria", "subcategoria"):
    linhas.append(("Superstore", cat, sub) + mapa_superstore(cat, sub))
for dep, cat in distintos("silver.dataco_itens", "department_name", "category_name"):
    linhas.append(("DataCo", dep, cat) + DATACO.get(cat, (OUT, "Outros")))
for (cat,) in distintos("silver.olist_itens", "categoria_en"):
    if cat is not None:
        linhas.append(("Olist", cat, None) + OLIST.get(cat, (OUT, "Outros")))
for (seg,) in distintos("silver.marketing_funnel", "business_segment"):
    if seg is not None:
        linhas.append(("Marketing Funnel", seg, None) + OLIST.get(seg, (OUT, "Outros")))
for (cat,) in distintos("silver.datafiniti_produtos", "categoria_principal"):
    if cat is not None:
        linhas.append(("Datafiniti", cat, None, ELE, categoria_eletronica(cat)))
for (sub,) in distintos("silver.amazon_produtos", "subcategoria"):
    if sub is not None:
        linhas.append(("Amazon", sub, None, ELE, categoria_eletronica(sub)))
# Roupa: Women's Clothing (departamento e classe) e Rent the Runway (departamento equivalente do Women's Clothing)
for dep, cla in distintos("silver.produtos", "department_name", "class_name"):
    linhas.append(("Womens Clothing", dep, cla, VES, "Roupa de Mulher"))
for (dep,) in distintos("silver.produtos_rtr", "departamento_wc"):
    linhas.append(("Rent the Runway", dep, None, VES, "Roupa de Mulher"))

mapa = (spark.createDataFrame(linhas, "fonte STRING, categoria_origem STRING, subcategoria_origem STRING, area STRING, categoria STRING")
    .withColumn("subcategoria", F.coalesce("subcategoria_origem", "categoria_origem")))

assert mapa.count() == mapa.select("fonte", "categoria_origem", "subcategoria_origem").distinct().count(), "categoria de origem repetida"
print("Linhas na mapa_categorias:", mapa.count())
mapa.groupBy("fonte").count().orderBy("fonte").show()
mapa.groupBy("area", "categoria").agg(F.collect_set("fonte").alias("fontes"), F.count("*").alias("categorias_origem")) \
    .orderBy("area", "categoria").show(60, truncate=False)

Linhas na mapa_categorias: 417
+----------------+-----+
|           fonte|count|
+----------------+-----+
|          Amazon|   16|
|          DataCo|   45|
|      Datafiniti|  224|
|Marketing Funnel|   18|
|           Olist|   71|
| Rent the Runway|    5|
|      Superstore|   17|
| Womens Clothing|   21|
+----------------+-----+

+-----------------------+-------------------------+-----------------------------------------------------------------+-----------------+
|area                   |categoria                |fontes                                                           |categorias_origem|
+-----------------------+-------------------------+-----------------------------------------------------------------+-----------------+
|Alimentação & Bebidas  |Alimentação & Bebidas    |[Olist]                                                          |3                |
|Animais                |Animais                  |[Olist, DataCo, Marketing Funnel]                                |3      

In [5]:
# Categorias que ficaram em "Outros": convém revê-las
mapa.filter(F.col("area") == "Outros").orderBy("fonte").show(40, truncate=False)
gravar_silver(mapa, "mapa_categorias")

+------+------------------------------+-------------------+------+---------+------------------------------+
|fonte |categoria_origem              |subcategoria_origem|area  |categoria|subcategoria                  |
+------+------------------------------+-------------------+------+---------+------------------------------+
|DataCo|Footwear                      |As Seen on TV!     |Outros|Outros   |As Seen on TV!                |
|DataCo|Outdoors                      |Trade-In           |Outros|Outros   |Trade-In                      |
|Olist |cool_stuff                    |null               |Outros|Outros   |cool_stuff                    |
|Olist |industry_commerce_and_business|null               |Outros|Outros   |industry_commerce_and_business|
|Olist |agro_industry_and_commerce    |null               |Outros|Outros   |agro_industry_and_commerce    |
|Olist |security_and_services         |null               |Outros|Outros   |security_and_services         |
|Olist |market_place        

## Países

In [6]:
# DataCo (espanhol) -> nome do World Data
DATACO_PAISES = {
    "Afganistán": "Afghanistan", "Alemania": "Germany", "Argentina": "Argentina", "Australia": "Australia", "Austria": "Austria",
    "Bangladés": "Bangladesh", "Bolivia": "Bolivia", "Brasil": "Brazil", "Bután": "Bhutan", "Bélgica": "Belgium",
    "Camboya": "Cambodia", "Chile": "Chile", "China": "China", "Chipre": "Cyprus", "Colombia": "Colombia",
    "Corea del Sur": "South Korea", "Cuba": "Cuba", "Dinamarca": "Denmark", "Ecuador": "Ecuador", "El Salvador": "El Salvador",
    "España": "Spain", "Filipinas": "Philippines", "Finlandia": "Finland", "Francia": "France", "Grecia": "Greece",
    "Guatemala": "Guatemala", "Haití": "Haiti", "Honduras": "Honduras", "Hong Kong": "Hong Kong", "India": "India",
    "Indonesia": "Indonesia", "Irlanda": "Republic of Ireland", "Italia": "Italy", "Jamaica": "Jamaica", "Japón": "Japan",
    "Luxemburgo": "Luxembourg", "Malasia": "Malaysia", "Mongolia": "Mongolia", "Myanmar (Birmania)": "Myanmar",
    "México": "Mexico", "Nepal": "Nepal", "Nicaragua": "Nicaragua", "Noruega": "Norway", "Pakistán": "Pakistan",
    "Panamá": "Panama", "Papúa Nueva Guinea": "Papua New Guinea", "Países Bajos": "Netherlands", "Perú": "Peru",
    "Portugal": "Portugal", "Reino Unido": "United Kingdom", "República Dominicana": "Dominican Republic",
    "Singapur": "Singapore", "Sri Lanka": "Sri Lanka", "Suecia": "Sweden", "Suiza": "Switzerland", "Tailandia": "Thailand",
    "Taiwán": "Taiwan", "Trinidad y Tobago": "Trinidad and Tobago", "Venezuela": "Venezuela", "Vietnam": "Vietnam",
}
# Superstore: só os nomes diferentes dos do World Data
SUPERSTORE_PAISES = {"Côte-d'Ivoire": "Ivory Coast", "Ireland": "Republic of Ireland", "Macedonia": "North Macedonia",
                     "Myanmar (Burma)": "Myanmar"}
# Códigos ISO em falta no World Data e territórios que não estão no World Data (ficam sem indicadores)
ISO_EM_FALTA = {"Eswatini": "SZ", "Namibia": "NA", "North Macedonia": "MK", "Palestinian National Authority": "PS",
                "Republic of Ireland": "IE", "Republic of the Congo": "CG", "Vatican City": "VA",
                "Hong Kong": "HK", "Taiwan": "TW", "French Guiana": "GF", "Martinique": "MQ"}

world = {r["pais"]: r["codigo_iso"] for r in spark.table("silver.world_data_2017").select("pais", "codigo_iso").collect()}

linhas = []   # (fonte, nome_origem, pais, codigo_iso, tem_indicadores)
def junta(fonte, nome, pais):
    iso = world.get(pais) or ISO_EM_FALTA.get(pais)
    linhas.append((fonte, nome, pais, iso, pais in world))

for pais in world:
    junta("World Data", pais, pais)
for (nome,) in distintos("silver.superstore_vendas", "pais"):
    junta("Superstore", nome, SUPERSTORE_PAISES.get(nome, nome))
for (nome,) in distintos("silver.dataco_itens", "order_country"):
    assert nome in DATACO_PAISES, f"País do DataCo sem tradução: {nome}"
    junta("DataCo", nome, DATACO_PAISES[nome])
for fonte in ["Olist", "INMET", "Marketing Funnel"]:
    junta(fonte, "Brasil", "Brazil")

paises = spark.createDataFrame(linhas, "fonte STRING, nome_origem STRING, pais STRING, codigo_iso STRING, tem_indicadores BOOLEAN")
assert paises.filter(F.col("codigo_iso").isNull()).count() == 0, "país sem código ISO"
assert paises.count() == paises.select("fonte", "nome_origem").distinct().count()
print("Linhas:", paises.count())
paises.groupBy("fonte").agg(F.count("*").alias("nomes"), F.sum((~F.col("tem_indicadores")).cast("int")).alias("sem_indicadores_world_data")).show()
paises.filter(~F.col("tem_indicadores")).show(truncate=False)
gravar_silver(paises, "paises")

Linhas: 393
+----------------+-----+--------------------------+
|           fonte|nomes|sem_indicadores_world_data|
+----------------+-----+--------------------------+
|      World Data|  195|                         0|
|      Superstore|  135|                         3|
|          DataCo|   60|                         2|
|Marketing Funnel|    1|                         0|
|           Olist|    1|                         0|
|           INMET|    1|                         0|
+----------------+-----+--------------------------+

+----------+-------------+-------------+----------+---------------+
|fonte     |nome_origem  |pais         |codigo_iso|tem_indicadores|
+----------+-------------+-------------+----------+---------------+
|Superstore|Martinique   |Martinique   |MQ        |false          |
|Superstore|Hong Kong    |Hong Kong    |HK        |false          |
|Superstore|French Guiana|French Guiana|GF        |false          |
|DataCo    |Taiwán       |Taiwan       |TW        |false   

## Verificação: ligação entre fontes

In [7]:
# Países que aparecem em mais de uma fonte (é por aqui que a dim_localizacao liga Superstore, DataCo e Olist)
spark.sql("""
    SELECT pais, codigo_iso, collect_set(fonte) AS fontes
    FROM silver.paises WHERE fonte <> 'World Data'
    GROUP BY pais, codigo_iso HAVING size(collect_set(fonte)) > 1 ORDER BY pais
""").show(60, truncate=False)

# Áreas comuns a mais de uma fonte (é por aqui que a dim_categoria_produto liga os datasets)
spark.sql("""
    SELECT area, collect_set(fonte) AS fontes FROM silver.mapa_categorias GROUP BY area ORDER BY area
""").show(truncate=False)

+-------------------+----------+----------------------------------------------------+
|pais               |codigo_iso|fontes                                              |
+-------------------+----------+----------------------------------------------------+
|Afghanistan        |AF        |[Superstore, DataCo]                                |
|Argentina          |AR        |[Superstore, DataCo]                                |
|Australia          |AU        |[Superstore, DataCo]                                |
|Austria            |AT        |[Superstore, DataCo]                                |
|Bangladesh         |BD        |[Superstore, DataCo]                                |
|Belgium            |BE        |[Superstore, DataCo]                                |
|Bolivia            |BO        |[Superstore, DataCo]                                |
|Brazil             |BR        |[Olist, Superstore, DataCo, Marketing Funnel, INMET]|
|Cambodia           |KH        |[Superstore, DataCo]  

In [8]:
spark.stop()